# 05 Model Comparison

?? notebook ????????? metrics CSV ??????????????????????????????????????

???????????????????????????????????????2???????

1. ????????????: ?????????????????????????????????
2. ????????????????: `baseline_m4` ??????? test ????????????????????

???????????????????? unconditional ? conditional ?????

In [1]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "Transport_amount_project" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

METRICS_DIR = PROJECT_ROOT / "output" / "forecasts" / "metrics"
FIGURES_DIR = PROJECT_ROOT / "output" / "forecasts" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Metrics dir:", METRICS_DIR)
print("Figures dir:", FIGURES_DIR)

Project root: C:\Users\fugat\Desktop\python_project\Transport_amount_project
Metrics dir: C:\Users\fugat\Desktop\python_project\Transport_amount_project\output\forecasts\metrics
Figures dir: C:\Users\fugat\Desktop\python_project\Transport_amount_project\output\forecasts\figures


## 1. ?? metrics ?????

??????????????????? skip ?????????????????????????????????????????

In [2]:
metric_files = {
    "naive": METRICS_DIR / "naive_metrics.csv",
    "sarimax_fixed": METRICS_DIR / "sarimax_metrics.csv",
    "sarimax_grid_best": METRICS_DIR / "sarimax_grid_best_models.csv",
    "ssm": METRICS_DIR / "ssm_metrics.csv",
    "prophet": METRICS_DIR / "prophet_metrics.csv",
    "autoformer_lite": METRICS_DIR / "autoformer_lite_metrics.csv",
    "prophet_regressors": METRICS_DIR / "prophet_regressors_metrics.csv",
}

loaded = {}
skipped = []
for key, path in metric_files.items():
    if path.exists():
        loaded[key] = pd.read_csv(path)
        print(f"loaded: {path.name} ({len(loaded[key])} rows)")
    else:
        skipped.append({"source": key, "path": str(path), "reason": "file not found"})

if skipped:
    display(pd.DataFrame(skipped))
else:
    print("No metric files were skipped.")

loaded: naive_metrics.csv (4 rows)
loaded: sarimax_metrics.csv (4 rows)
loaded: sarimax_grid_best_models.csv (4 rows)
loaded: ssm_metrics.csv (2 rows)
loaded: prophet_metrics.csv (2 rows)
loaded: autoformer_lite_metrics.csv (1 rows)
loaded: prophet_regressors_metrics.csv (1 rows)
No metric files were skipped.


## 2. ????????

????????????? `information_set = unconditional` ???SARIMAX/SSM/Prophet regressors ???? test ???????????????????? `conditional_exog_known` ????

In [3]:
notes = {
    "naive": "Naive baseline; no exogenous variables.",
    "seasonal_naive": "Seasonal naive baseline; no exogenous variables; recursive for horizons beyond 12 months.",
    "sarima_fixed": "Fixed-order SARIMA; no exogenous variables.",
    "sarima_grid_best": "Small SARIMA order grid best by converged RMSE; no exogenous variables.",
    "prophet": "Prophet on log scale; no holidays and no regressors.",
    "autoformer_lite": "Autoformer-inspired / decomposition Transformer baseline, not a strict Autoformer; no exogenous variables; fixed B only; reference baseline for small monthly sample.",
    "sarimax_fixed": "Fixed-order SARIMAX; conditional forecast with test-period baseline_m4 exogenous dummies known.",
    "sarimax_grid_best": "Small SARIMAX order grid best by converged RMSE; conditional forecast with test-period baseline_m4 exogenous dummies known.",
    "ssm_conditional": "Main-analysis SSM; conditional forecast with test-period baseline_m4 exogenous dummies known.",
    "prophet_regressors": "Prophet with baseline_m4 regressors; conditional forecast; fixed B only; holidays???weekly/daily seasonality??.",
}

unconditional_models = {
    "naive",
    "seasonal_naive",
    "sarima_fixed",
    "sarima_grid_best",
    "prophet",
    "autoformer_lite",
}
conditional_models = {
    "sarimax_fixed",
    "sarimax_grid_best",
    "ssm_conditional",
    "prophet_regressors",
}


def information_set_for(model: str) -> str:
    if model in conditional_models:
        return "conditional_exog_known"
    return "unconditional"


def forecast_type_for(model: str) -> str:
    if model in conditional_models:
        return "conditional"
    return "unconditional"


def add_rows(rows: list[dict], df: pd.DataFrame, model_name: str, source_file: str) -> None:
    for _, row in df.iterrows():
        rows.append(
            {
                "split": row["split"],
                "model": model_name,
                "forecast_type": forecast_type_for(model_name),
                "information_set": information_set_for(model_name),
                "spec_name": row.get("spec_name", ""),
                "rmse": row["rmse"],
                "mae": row["mae"],
                "mape": row["mape"],
                "mase": row["mase"],
                "source_file": source_file,
                "notes": notes[model_name],
            }
        )

## 3. ??????

? metrics ??????????`sarima`/`sarimax` ? fixed-order ? grid-best ???????????

In [4]:
rows: list[dict] = []

if "naive" in loaded:
    for model_name in ["naive", "seasonal_naive"]:
        add_rows(rows, loaded["naive"][loaded["naive"]["model"] == model_name], model_name, "naive_metrics.csv")

if "sarimax_fixed" in loaded:
    fixed_df = loaded["sarimax_fixed"]
    add_rows(rows, fixed_df[fixed_df["model"] == "sarima"], "sarima_fixed", "sarimax_metrics.csv")
    add_rows(rows, fixed_df[fixed_df["model"] == "sarimax"], "sarimax_fixed", "sarimax_metrics.csv")

if "sarimax_grid_best" in loaded:
    grid_df = loaded["sarimax_grid_best"]
    add_rows(rows, grid_df[grid_df["model"] == "sarima"], "sarima_grid_best", "sarimax_grid_best_models.csv")
    add_rows(rows, grid_df[grid_df["model"] == "sarimax"], "sarimax_grid_best", "sarimax_grid_best_models.csv")

if "ssm" in loaded:
    add_rows(rows, loaded["ssm"], "ssm_conditional", "ssm_metrics.csv")

if "prophet" in loaded:
    add_rows(rows, loaded["prophet"], "prophet", "prophet_metrics.csv")

if "autoformer_lite" in loaded:
    add_rows(rows, loaded["autoformer_lite"], "autoformer_lite", "autoformer_lite_metrics.csv")

if "prophet_regressors" in loaded:
    add_rows(rows, loaded["prophet_regressors"], "prophet_regressors", "prophet_regressors_metrics.csv")

comparison = pd.DataFrame(rows)
metric_cols = ["rmse", "mae", "mape", "mase"]
comparison[metric_cols] = comparison[metric_cols].apply(pd.to_numeric, errors="coerce")
comparison = comparison.sort_values(["split", "information_set", "rmse", "model"]).reset_index(drop=True)

comparison["rmse_rank_within_split"] = comparison.groupby("split")["rmse"].rank(method="min")
comparison["mape_rank_within_split"] = comparison.groupby("split")["mape"].rank(method="min")
comparison["mase_rank_within_split"] = comparison.groupby("split")["mase"].rank(method="min")
comparison["rmse_rank_within_information_set"] = comparison.groupby(["split", "information_set"])["rmse"].rank(method="min")
comparison["mape_rank_within_information_set"] = comparison.groupby(["split", "information_set"])["mape"].rank(method="min")
comparison["mase_rank_within_information_set"] = comparison.groupby(["split", "information_set"])["mase"].rank(method="min")

ordered_cols = [
    "split", "model", "forecast_type", "information_set", "spec_name",
    "rmse", "mae", "mape", "mase",
    "rmse_rank_within_split", "mape_rank_within_split", "mase_rank_within_split",
    "rmse_rank_within_information_set", "mape_rank_within_information_set", "mase_rank_within_information_set",
    "source_file", "notes",
]
comparison = comparison[ordered_cols]
comparison.to_csv(METRICS_DIR / "model_comparison_fixed_splits.csv", index=False)

display(comparison)

,split,model,forecast_type,information_set,spec_name,rmse,mae,mape,mase,rmse_rank_within_split,mape_rank_within_split,mase_rank_within_split,rmse_rank_within_information_set,mape_rank_within_information_set,mase_rank_within_information_set,source_file,notes
0,fixed_a,sarimax_grid_best,conditional,conditional_exog_known,baseline_m4,29828.573653,26984.723325,6.782756,2.491939,2.0,2.0,2.0,1.0,1.0,1.0,sarimax_grid_best_models.csv,Small SARIMAX order grid best by converged RMS...
1,fixed_a,ssm_conditional,conditional,conditional_exog_known,baseline_m4,30857.609560,27827.594523,7.017864,2.569775,3.0,3.0,3.0,2.0,2.0,2.0,ssm_metrics.csv,Main-analysis SSM; conditional forecast with t...
2,fixed_a,sarimax_fixed,conditional,conditional_exog_known,baseline_m4,33985.764338,31233.125826,7.829297,2.884263,4.0,4.0,4.0,3.0,3.0,3.0,sarimax_metrics.csv,Fixed-order SARIMAX; conditional forecast with...
3,fixed_a,prophet,unconditional,unconditional,prophet_no_regressors,27498.765818,24252.685218,6.245448,2.239645,1.0,1.0,1.0,1.0,1.0,1.0,prophet_metrics.csv,Prophet on log scale; no holidays and no regre...
4,fixed_a,sarima_grid_best,unconditional,unconditional,none,40578.640251,38202.371671,9.594779,3.527847,5.0,5.0,5.0,2.0,2.0,2.0,sarimax_grid_best_models.csv,Small SARIMA order grid best by converged RMSE...
5,fixed_a,sarima_fixed,unconditional,unconditional,none,41548.023660,39180.565637,9.840238,3.618180,6.0,6.0,6.0,3.0,3.0,3.0,sarimax_metrics.csv,Fixed-order SARIMA; no exogenous variables.
6,fixed_a,seasonal_naive,unconditional,unconditional,baseline_m4,44753.891938,41456.250000,10.335366,3.828331,7.0,7.0,7.0,4.0,4.0,4.0,naive_metrics.csv,Seasonal naive baseline; no exogenous variable...
7,fixed_a,naive,unconditional,unconditional,baseline_m4,80656.774506,75560.833333,20.088095,6.977762,8.0,8.0,8.0,5.0,5.0,5.0,naive_metrics.csv,Naive baseline; no exogenous variables.
8,fixed_b,sarimax_grid_best,conditional,conditional_exog_known,baseline_m4,7975.062046,6701.292189,1.709270,0.535811,1.0,1.0,1.0,1.0,1.0,1.0,sarimax_grid_best_models.csv,Small SARIMAX order grid best by converged RMS...
9,fixed_b,ssm_conditional,conditional,conditional_exog_known,baseline_m4,8112.663937,6756.339867,1.718500,0.540212,2.0,2.0,2.0,2.0,2.0,2.0,ssm_metrics.csv,Main-analysis SSM; conditional forecast with t...


## 4. ????????????

?????????????????????????????????????????? fixed A/B ??? unconditional ??????????

In [5]:
unconditional = comparison[comparison["information_set"] == "unconditional"].copy()
unconditional = unconditional.sort_values(["split", "rmse", "model"]).reset_index(drop=True)
unconditional.to_csv(METRICS_DIR / "model_comparison_unconditional.csv", index=False)
display(unconditional)

,split,model,forecast_type,information_set,spec_name,rmse,mae,mape,mase,rmse_rank_within_split,mape_rank_within_split,mase_rank_within_split,rmse_rank_within_information_set,mape_rank_within_information_set,mase_rank_within_information_set,source_file,notes
0,fixed_a,prophet,unconditional,unconditional,prophet_no_regressors,27498.765818,24252.685218,6.245448,2.239645,1.0,1.0,1.0,1.0,1.0,1.0,prophet_metrics.csv,Prophet on log scale; no holidays and no regre...
1,fixed_a,sarima_grid_best,unconditional,unconditional,none,40578.640251,38202.371671,9.594779,3.527847,5.0,5.0,5.0,2.0,2.0,2.0,sarimax_grid_best_models.csv,Small SARIMA order grid best by converged RMSE...
2,fixed_a,sarima_fixed,unconditional,unconditional,none,41548.023660,39180.565637,9.840238,3.618180,6.0,6.0,6.0,3.0,3.0,3.0,sarimax_metrics.csv,Fixed-order SARIMA; no exogenous variables.
3,fixed_a,seasonal_naive,unconditional,unconditional,baseline_m4,44753.891938,41456.250000,10.335366,3.828331,7.0,7.0,7.0,4.0,4.0,4.0,naive_metrics.csv,Seasonal naive baseline; no exogenous variable...
4,fixed_a,naive,unconditional,unconditional,baseline_m4,80656.774506,75560.833333,20.088095,6.977762,8.0,8.0,8.0,5.0,5.0,5.0,naive_metrics.csv,Naive baseline; no exogenous variables.
5,fixed_b,seasonal_naive,unconditional,unconditional,post2020_m5,13622.783591,11619.923077,2.906761,0.929087,4.0,4.0,4.0,1.0,1.0,1.0,naive_metrics.csv,Seasonal naive baseline; no exogenous variable...
6,fixed_b,sarima_grid_best,unconditional,unconditional,none,14790.762992,12236.132613,3.071175,0.978357,5.0,5.0,5.0,2.0,2.0,2.0,sarimax_grid_best_models.csv,Small SARIMA order grid best by converged RMSE...
7,fixed_b,sarima_fixed,unconditional,unconditional,none,16686.680778,13964.593736,3.502203,1.116558,6.0,7.0,7.0,3.0,3.0,3.0,sarimax_metrics.csv,Fixed-order SARIMA; no exogenous variables.
8,fixed_b,prophet,unconditional,unconditional,prophet_no_regressors,26730.003403,19845.066371,4.834269,1.586739,8.0,8.0,8.0,4.0,4.0,4.0,prophet_metrics.csv,Prophet on log scale; no holidays and no regre...
9,fixed_b,autoformer_lite,unconditional,unconditional,autoformer_lite_univariate,35587.389423,32410.636313,8.536476,2.591436,9.0,9.0,9.0,5.0,5.0,5.0,autoformer_lite_metrics.csv,Autoformer-inspired / decomposition Transforme...


## 5. ????????????????

?????`baseline_m4` ??????? test ???????????? conditional forecast ??????SARIMAX/SSM/Prophet regressors ???????????

In [6]:
conditional = comparison[comparison["information_set"] == "conditional_exog_known"].copy()
conditional = conditional.sort_values(["split", "rmse", "model"]).reset_index(drop=True)
conditional.to_csv(METRICS_DIR / "model_comparison_conditional.csv", index=False)

fixed_b_conditional = conditional[conditional["split"] == "fixed_b"].copy().reset_index(drop=True)
fixed_b_conditional.to_csv(METRICS_DIR / "model_comparison_fixed_b_conditional.csv", index=False)

display(conditional)
print("fixed B conditional models")
display(fixed_b_conditional)

,split,model,forecast_type,information_set,spec_name,rmse,mae,mape,mase,rmse_rank_within_split,mape_rank_within_split,mase_rank_within_split,rmse_rank_within_information_set,mape_rank_within_information_set,mase_rank_within_information_set,source_file,notes
0,fixed_a,sarimax_grid_best,conditional,conditional_exog_known,baseline_m4,29828.573653,26984.723325,6.782756,2.491939,2.0,2.0,2.0,1.0,1.0,1.0,sarimax_grid_best_models.csv,Small SARIMAX order grid best by converged RMS...
1,fixed_a,ssm_conditional,conditional,conditional_exog_known,baseline_m4,30857.609560,27827.594523,7.017864,2.569775,3.0,3.0,3.0,2.0,2.0,2.0,ssm_metrics.csv,Main-analysis SSM; conditional forecast with t...
2,fixed_a,sarimax_fixed,conditional,conditional_exog_known,baseline_m4,33985.764338,31233.125826,7.829297,2.884263,4.0,4.0,4.0,3.0,3.0,3.0,sarimax_metrics.csv,Fixed-order SARIMAX; conditional forecast with...
3,fixed_b,sarimax_grid_best,conditional,conditional_exog_known,baseline_m4,7975.062046,6701.292189,1.709270,0.535811,1.0,1.0,1.0,1.0,1.0,1.0,sarimax_grid_best_models.csv,Small SARIMAX order grid best by converged RMS...
4,fixed_b,ssm_conditional,conditional,conditional_exog_known,baseline_m4,8112.663937,6756.339867,1.718500,0.540212,2.0,2.0,2.0,2.0,2.0,2.0,ssm_metrics.csv,Main-analysis SSM; conditional forecast with t...
5,fixed_b,sarimax_fixed,conditional,conditional_exog_known,baseline_m4,8196.974613,6873.290432,1.758244,0.549563,3.0,3.0,3.0,3.0,3.0,3.0,sarimax_metrics.csv,Fixed-order SARIMAX; conditional forecast with...
6,fixed_b,prophet_regressors,conditional,conditional_exog_known,baseline_m4,17877.060334,13052.309116,3.201742,1.043615,7.0,6.0,6.0,4.0,4.0,4.0,prophet_regressors_metrics.csv,Prophet with baseline_m4 regressors; condition...


fixed B conditional models


,split,model,forecast_type,information_set,spec_name,rmse,mae,mape,mase,rmse_rank_within_split,mape_rank_within_split,mase_rank_within_split,rmse_rank_within_information_set,mape_rank_within_information_set,mase_rank_within_information_set,source_file,notes
0,fixed_b,sarimax_grid_best,conditional,conditional_exog_known,baseline_m4,7975.062046,6701.292189,1.709270,0.535811,1.0,1.0,1.0,1.0,1.0,1.0,sarimax_grid_best_models.csv,Small SARIMAX order grid best by converged RMS...
1,fixed_b,ssm_conditional,conditional,conditional_exog_known,baseline_m4,8112.663937,6756.339867,1.718500,0.540212,2.0,2.0,2.0,2.0,2.0,2.0,ssm_metrics.csv,Main-analysis SSM; conditional forecast with t...
2,fixed_b,sarimax_fixed,conditional,conditional_exog_known,baseline_m4,8196.974613,6873.290432,1.758244,0.549563,3.0,3.0,3.0,3.0,3.0,3.0,sarimax_metrics.csv,Fixed-order SARIMAX; conditional forecast with...
3,fixed_b,prophet_regressors,conditional,conditional_exog_known,baseline_m4,17877.060334,13052.309116,3.201742,1.043615,7.0,6.0,6.0,4.0,4.0,4.0,prophet_regressors_metrics.csv,Prophet with baseline_m4 regressors; condition...


## 6. split ??? best model

`best_overall_reference` ?????????????????????????????????????????????? `best_unconditional` ? `best_conditional` ???????

In [7]:
best_rows = []
for split_name in sorted(comparison["split"].unique()):
    split_df = comparison[comparison["split"] == split_name]
    uncond_df = split_df[split_df["information_set"] == "unconditional"]
    cond_df = split_df[split_df["information_set"] == "conditional_exog_known"]
    overall_df = split_df

    row = {"split": split_name}
    if not uncond_df.empty:
        best = uncond_df.sort_values("rmse").iloc[0]
        row.update(
            {
                "best_unconditional": best["model"],
                "best_unconditional_rmse": best["rmse"],
                "best_unconditional_mape": best["mape"],
                "best_unconditional_mase": best["mase"],
            }
        )
    if not cond_df.empty:
        best = cond_df.sort_values("rmse").iloc[0]
        row.update(
            {
                "best_conditional": best["model"],
                "best_conditional_rmse": best["rmse"],
                "best_conditional_mape": best["mape"],
                "best_conditional_mase": best["mase"],
            }
        )
    if not overall_df.empty:
        best = overall_df.sort_values("rmse").iloc[0]
        row.update(
            {
                "best_overall_reference": best["model"],
                "best_overall_reference_information_set": best["information_set"],
                "best_overall_reference_rmse": best["rmse"],
                "best_overall_reference_mape": best["mape"],
                "best_overall_reference_mase": best["mase"],
            }
        )
    best_rows.append(row)

best_by_split = pd.DataFrame(best_rows)
best_by_split.to_csv(METRICS_DIR / "model_comparison_best_by_split.csv", index=False)
display(best_by_split)

,split,best_unconditional,best_unconditional_rmse,best_unconditional_mape,best_unconditional_mase,best_conditional,best_conditional_rmse,best_conditional_mape,best_conditional_mase,best_overall_reference,best_overall_reference_information_set,best_overall_reference_rmse,best_overall_reference_mape,best_overall_reference_mase
0,fixed_a,prophet,27498.765818,6.245448,2.239645,sarimax_grid_best,29828.573653,6.782756,2.491939,prophet,unconditional,27498.765818,6.245448,2.239645
1,fixed_b,seasonal_naive,13622.783591,2.906761,0.929087,sarimax_grid_best,7975.062046,1.709270,0.535811,sarimax_grid_best,conditional_exog_known,7975.062046,1.709270,0.535811


## 7. RMSE ?

unconditional ? conditional ?????????fixed A ?????? autoformer_lite / prophet_regressors ?????????

In [8]:
def plot_rmse_by_split(df: pd.DataFrame, title: str, output_path: Path) -> None:
    plot_df = df.copy()
    if plot_df.empty:
        print(f"No rows for {title}")
        return

    split_order = [split for split in ["fixed_a", "fixed_b"] if split in set(plot_df["split"])]
    model_order = (
        plot_df.groupby("model")["rmse"]
        .mean()
        .sort_values()
        .index.tolist()
    )
    pivot = plot_df.pivot_table(index="model", columns="split", values="rmse", aggfunc="first")
    pivot = pivot.reindex(index=model_order, columns=split_order)

    ax = pivot.plot(kind="bar", figsize=(11, 5.5), width=0.78)
    ax.set_title(title)
    ax.set_xlabel("Model")
    ax.set_ylabel("RMSE")
    ax.grid(axis="y", color="0.85", linewidth=0.8)
    ax.legend(title="split")
    ax.tick_params(axis="x", rotation=35)
    for label in ax.get_xticklabels():
        label.set_horizontalalignment("right")
    ax.figure.tight_layout()
    ax.figure.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(ax.figure)

plot_rmse_by_split(
    unconditional,
    "Unconditional forecast benchmarks: RMSE by split",
    FIGURES_DIR / "model_comparison_unconditional_rmse.png",
)
plot_rmse_by_split(
    conditional,
    "Conditional exogenous-known models: RMSE by split",
    FIGURES_DIR / "model_comparison_conditional_rmse.png",
)

print("Saved RMSE figures.")

Saved RMSE figures.


## 8. ??????

- ????????????????????????????????????????????????????????????????????????????????????????
- ???????????????fixed A ? Prophet ??????fixed B ? seasonal naive ?????????? fixed B ?????????????????
- ???????????????????fixed B ? SARIMAX grid best ??????SSM ???????????????????????????????SARIMAX/SSM ?????????
- prophet_regressors ??????? Prophet ????????SSM/SARIMAX ?????????Prophet ??? `baseline_m4` regressors ????????????? SARIMAX ????????? fixed B ???????????????
- fixed A ?? Prophet regressors ?????????fixed A ?????? 2019-12 ??????`covid_main`, `covid_wave1`, `covid_2021`, `post_stat_change` ???????????????? fixed B ??????
- SSM ????????????????? Prophet regressors ????????????????????????SSM ???????????????????conditional forecast ????????????????
- conditional forecast ? test ??????????????????????unconditional forecast ???????????????